# ParaFrame Demo

This notebook demonstrates the current `hallmark` ParaFrame API.

`ParaFrame` is a Pandas DataFrame subclass that discovers and parses files whose names follow a Python format string pattern. It extracts named parameters directly into DataFrame columns, making it easy to filter and select files across large parameter surveys.

Key components:
- **`.hallmark.yaml`** — config file that lives alongside your data, declaring format strings and optional regex encodings for special filename conventions
- **`ParaFrame.parse()`** — discovers matching files and returns a DataFrame
- **`ParaFrame.__call__()`** — filters the DataFrame by column values

In [1]:
# auto reload modules when they change
%load_ext autoreload
%autoreload 2

## 1. The `.hallmark.yaml` Config File

Before calling `ParaFrame.parse()`, a `.hallmark.yaml` must exist in your data directory. It declares:
- `fmt`: the Python format string for your file naming convention
- `encoding` (optional): regex patterns for parameters that use special filename conventions

Here is the `.hallmark.yaml` used in this demo:

In [ ]:
! cat data/.hallmark.yaml  # check what fmts are present

data:
- fmt: '{mag:d}a{aspin}_w{win:d}.h5'
  # path_to_fmt: m5/data
  encoding:
    aspin: m([0-9]+(\.[0-9]+)?|\.[0-9]+)

- fmt: a_{a:d}/b_{b:d}.txt
  # path_to_fmt: data

- fmt: a{aspin}/b_{b:d}.txt
  # path_to_fmt: data
  encoding:
    aspin: ''


## 2. Simple File Discovery (no encoding)

Files structured as `a_{a:d}/b_{b:d}.txt` — no special encoding, so `encoding=False` (the default).

### Create sample data files

In [3]:
%%bash
# make some test files to parse
for a in {0..4}; do
  mkdir -p "data/a_$a"
  for b in {10..14}; do
    touch "data/a_$a/b_$b.txt"
  done
done
ls data/a_0/

b_10.txt
b_11.txt
b_12.txt
b_13.txt
b_14.txt


### Parse with `repo_path`

`repo_path` tells `ParaFrame` where to find `.hallmark.yaml` and where to search for files.

In [4]:
from hallmark import ParaFrame

# repo_path tells hallmark where to find .hallmark.yaml
pf = ParaFrame.parse("a_{a:d}/b_{b:d}.txt", repo_path="data")
pf

Pattern: data/a_{a:d}/b_{b:d}.txt


,path,a,b
0,a_0/b_10.txt,0.0,10.0
1,a_0/b_11.txt,0.0,11.0
2,a_0/b_12.txt,0.0,12.0
3,a_0/b_13.txt,0.0,13.0
4,a_0/b_14.txt,0.0,14.0
5,a_1/b_10.txt,1.0,10.0
6,a_1/b_11.txt,1.0,11.0
7,a_1/b_12.txt,1.0,12.0
8,a_1/b_13.txt,1.0,13.0
9,a_1/b_14.txt,1.0,14.0


### Parse with `set_rel_yaml_path`

Instead of passing `repo_path` every time, set it globally once.

In [5]:
from hallmark import ParaFrame, set_rel_yaml_path

# set once globally so you dont have to pass repo_path every time
set_rel_yaml_path("data")

pf = ParaFrame.parse("a_{a:d}/b_{b:d}.txt")
pf

Pattern: data/a_{a:d}/b_{b:d}.txt


,path,a,b
0,a_0/b_10.txt,0.0,10.0
1,a_0/b_11.txt,0.0,11.0
2,a_0/b_12.txt,0.0,12.0
3,a_0/b_13.txt,0.0,13.0
4,a_0/b_14.txt,0.0,14.0
5,a_1/b_10.txt,1.0,10.0
6,a_1/b_11.txt,1.0,11.0
7,a_1/b_12.txt,1.0,12.0
8,a_1/b_13.txt,1.0,13.0
9,a_1/b_14.txt,1.0,14.0


### Debug mode

`debug=True` prints the resolved glob pattern and match count — useful when no files are found.

In [6]:
# debug=True shows the resolved glob pattern and how many files matched
pf = ParaFrame.parse("a_{a:d}/b_{b:d}.txt", repo_path="data", debug=True)

Pattern: data/a_{a:d}/b_{b:d}.txt
0 data/a_{a:d}/b_{b:d}.txt () {}
1 data/a_{a:s}/b_{b:d}.txt () {'a': '*'}
2 data/a_{a:s}/b_{b:s}.txt () {'a': '*', 'b': '*'}
Pattern: "data/a_*/b_*.txt"
25 matches, e.g., "data/a_0/b_10.txt"


## 3. Filtering

Call a `ParaFrame` like a function to filter by column values:
- **scalar** → exact match
- **list or tuple** → OR match
- **chaining** → AND logic

In [7]:
# Filter a == 0
pf(a=0)

,path,a,b
0,a_0/b_10.txt,0.0,10.0
1,a_0/b_11.txt,0.0,11.0
2,a_0/b_12.txt,0.0,12.0
3,a_0/b_13.txt,0.0,13.0
4,a_0/b_14.txt,0.0,14.0


In [8]:
# Filter a == 0 or a == 1
pf(a=[0, 1])

,path,a,b
0,a_0/b_10.txt,0.0,10.0
1,a_0/b_11.txt,0.0,11.0
2,a_0/b_12.txt,0.0,12.0
3,a_0/b_13.txt,0.0,13.0
4,a_0/b_14.txt,0.0,14.0
5,a_1/b_10.txt,1.0,10.0
6,a_1/b_11.txt,1.0,11.0
7,a_1/b_12.txt,1.0,12.0
8,a_1/b_13.txt,1.0,13.0
9,a_1/b_14.txt,1.0,14.0


In [9]:
# Filter a == 0 AND b == 10
pf(a=0)(b=10)

,path,a,b
0,a_0/b_10.txt,0.0,10.0


In [10]:
# use pandas masking directly for more complex conditions
pf[(2 <= pf.a) & (pf.a <= 4)]

,path,a,b
10,a_2/b_10.txt,2.0,10.0
11,a_2/b_11.txt,2.0,11.0
12,a_2/b_12.txt,2.0,12.0
13,a_2/b_13.txt,2.0,13.0
14,a_2/b_14.txt,2.0,14.0
15,a_3/b_10.txt,3.0,10.0
16,a_3/b_11.txt,3.0,11.0
17,a_3/b_12.txt,3.0,12.0
18,a_3/b_13.txt,3.0,13.0
19,a_3/b_14.txt,3.0,14.0


### Using filtered paths in a workflow

In [11]:
# pf(a=0, b=10) means a==0 OR b==10
for p in pf(a=0, b=10).path:
    print(f'Processing "{p}"...')

Processing "a_0/b_10.txt"...
Processing "a_0/b_11.txt"...
Processing "a_0/b_12.txt"...
Processing "a_0/b_13.txt"...
Processing "a_0/b_14.txt"...
Processing "a_1/b_10.txt"...
Processing "a_2/b_10.txt"...
Processing "a_3/b_10.txt"...
Processing "a_4/b_10.txt"...


## 4. Encoding — Special Filename Conventions

Some datasets use non-standard filename conventions. For example, black hole spin values are stored as `m0.94` to represent `-0.94` (the `m` prefix means "minus").

The `.hallmark.yaml` handles this with the `encoding` key, which holds a regex pattern. When `encoding=True` is passed, the regex substitution is applied to the filename before parsing.

Relevant YAML entry:
```yaml
- fmt: '{mag:d}a{aspin}_w{win:d}.h5'
  encoding:
    aspin: m([0-9]+(\.[0-9]+)?|\.[0-9]+)
```

### Create sample spin data files

In [12]:
%%bash
# spin values use m prefix to mean negative e.g. m0.94 = -0.94
for mag in 1 2; do
  for aspin in 0 0.5 m0.5 m0.94; do
    for win in 10 20; do
      touch "data/${mag}a${aspin}_w${win}.h5"
    done
  done
done
ls data/*.h5

data/1a0_w10.h5
data/1a0_w20.h5
data/1a0.5_w10.h5
data/1a0.5_w20.h5
data/1am0.5_w10.h5
data/1am0.5_w

20.h5
data/1am0.94_w10.h5
data/1am0.94_w20.h5
data/2a0_w10.h5
data/2a0_w20.h5
data/2a0.5_w10.h5
data

/2a0.5_w20.h5
data/2am0.5_w10.h5
data/2am0.5_w20.h5
data/2am0.94_w10.h5
data/2am0.94_w20.h5


### Parse with `encoding=True`

The `m` prefix is decoded by the regex: `m0.94` → `-0.94`, `m0.5` → `-0.5`.

In [13]:
# encoding=True applies the regex from .hallmark.yaml before parsing
# so m0.94 in the filename becomes -0.94 in the dataframe
pf_spin = ParaFrame.parse("{mag:d}a{aspin}_w{win:d}.h5", repo_path="data", encoding=True)
pf_spin

Pattern: data/{mag:d}a{aspin}_w{win:d}.h5


,path,mag,aspin,win
0,1a0.5_w10.h5,1.0,0.50,10.0
1,1a0.5_w20.h5,1.0,0.50,20.0
2,1a0_w10.h5,1.0,0.00,10.0
3,1a0_w20.h5,1.0,0.00,20.0
4,1am0.5_w10.h5,1.0,-0.50,10.0
5,1am0.5_w20.h5,1.0,-0.50,20.0
6,1am0.94_w10.h5,1.0,-0.94,10.0
7,1am0.94_w20.h5,1.0,-0.94,20.0
8,2a0.5_w10.h5,2.0,0.50,10.0
9,2a0.5_w20.h5,2.0,0.50,20.0


The `aspin` column contains decoded values (e.g. `-0.94`) even though the filenames use `m0.94`. Filtering uses the decoded values:

In [14]:
# filter using the decoded float values not the raw filename strings
pf_spin(aspin=[-0.5, -0.94])

,path,mag,aspin,win
4,1am0.5_w10.h5,1.0,-0.50,10.0
5,1am0.5_w20.h5,1.0,-0.50,20.0
6,1am0.94_w10.h5,1.0,-0.94,10.0
7,1am0.94_w20.h5,1.0,-0.94,20.0
12,2am0.5_w10.h5,2.0,-0.50,10.0
13,2am0.5_w20.h5,2.0,-0.50,20.0
14,2am0.94_w10.h5,2.0,-0.94,10.0
15,2am0.94_w20.h5,2.0,-0.94,20.0


### Encoding validation

`ParaFrame` enforces consistency between your call and the YAML spec:
- If the YAML has a regex for a parameter, you **must** pass `encoding=True`
- If the YAML has no regex, you **must** use `encoding=False` (the default)

This prevents silent mis-parses.

In [15]:
# hallmark raises an error if you forget encoding=True when the yaml has a regex
try:
    ParaFrame.parse("{mag:d}a{aspin}_w{win:d}.h5", repo_path="data", encoding=False)
except ValueError as e:
    print(e)

Error: '{mag:d}a{aspin}_w{win:d}.h5' has a regex spec, 
                so you must use encoding=True
